# 小样本过拟合测试 — 数据集采样

从完整数据集中每类随机抽取指定数量的样本，保持原始目录结构输出。

数据集结构：
```
data_dir/
    label_0/
        ToT/
            xxx.txt
        ToA/
            xxx.txt
    label_1/
        ToT/
        ToA/
    ...
```

## 1. 配置

In [ ]:
from pathlib import Path

# TODO: 修改以下路径
INPUT_DIR = Path(r"d:\Project\Timepix\Data\Alpha_Clean")
OUTPUT_DIR = Path(r"d:\Project\Timepix\Data\Alpha_Clean_100")

SAMPLES_PER_CLASS = 100  # 每类保留的样本数
SEED = 42                # 随机种子，保证可复现

print(f"输入目录: {INPUT_DIR}")
print(f"输出目录: {OUTPUT_DIR}")
print(f"每类样本数: {SAMPLES_PER_CLASS}")

## 2. 扫描数据集结构

In [ ]:
import random
import shutil

random.seed(SEED)

# 自动发现所有标签子目录
label_dirs = sorted(
    [d for d in INPUT_DIR.iterdir() if d.is_dir()]
)

print(f"发现 {len(label_dirs)} 个类别:\n")

# 收集每个类别下的样本文件名
class_samples = {}  # {label_name: [stem1, stem2, ...]}

for label_dir in label_dirs:
    label_name = label_dir.name
    
    # 发现该标签下有哪些模态子目录
    modality_dirs = sorted([d for d in label_dir.iterdir() if d.is_dir()])
    modality_names = [d.name for d in modality_dirs]
    
    if not modality_dirs:
        print(f"  {label_name}: 无模态子目录，跳过")
        continue
    
    # 提示只有单模态的情况
    expected_modalities = {"ToT", "ToA"}
    found_modalities = set(modality_names)
    missing = expected_modalities - found_modalities
    if missing:
        print(f"  ⚠ 类别 '{label_name}' 缺少模态: {missing}，仅有 {found_modalities}")
    
    # 以第一个模态目录的文件名茎作为样本 ID 列表
    reference_dir = modality_dirs[0]
    stems = sorted([f.stem for f in reference_dir.glob("*.txt")])
    class_samples[label_name] = stems
    
    print(f"  {label_name}: {len(stems)} 个样本, 模态 = {modality_names}")

## 3. 随机采样并复制

In [ ]:
total_copied = 0

for label_dir in label_dirs:
    label_name = label_dir.name
    if label_name not in class_samples:
        continue
    
    stems = class_samples[label_name]
    n = min(SAMPLES_PER_CLASS, len(stems))
    
    if n < SAMPLES_PER_CLASS:
        print(f"⚠ 类别 '{label_name}' 仅有 {len(stems)} 个样本，不足 {SAMPLES_PER_CLASS}，全部保留")
    
    selected = random.sample(stems, n)
    
    # 发现所有模态子目录
    modality_dirs = [d for d in label_dir.iterdir() if d.is_dir()]
    
    copied_count = 0
    for stem in selected:
        for mod_dir in modality_dirs:
            src = mod_dir / f"{stem}.txt"
            if not src.exists():
                continue
            dst = OUTPUT_DIR / label_name / mod_dir.name / f"{stem}.txt"
            dst.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(src, dst)
        copied_count += 1
    
    total_copied += copied_count
    print(f"  {label_name}: 采样 {copied_count} 个样本")

print(f"\n总计复制 {total_copied} 个样本到 {OUTPUT_DIR}")

## 4. 验证输出

In [ ]:
print("输出数据集结构:\n")

for label_dir in sorted(OUTPUT_DIR.iterdir()):
    if not label_dir.is_dir():
        continue
    for mod_dir in sorted(label_dir.iterdir()):
        if not mod_dir.is_dir():
            continue
        count = len(list(mod_dir.glob("*.txt")))
        print(f"  {label_dir.name}/{mod_dir.name}: {count} 个文件")